In [4]:
import numpy as np
import time
import pandas as pd
from scipy import stats
import warnings
from bezierv.classes.distfit import DistFit
from tqdm import tqdm

In [10]:
def test_bezierv_fit_distributions(m: int, n: int = 5, n_trials: int = 100, output_file: str = 'benchmark_results.csv'):
    """
    Benchmarks bezierv fits against standard statistical distributions over multiple trials.
    Includes a percentage progress bar.
    """
    
    warnings.filterwarnings("ignore")
    
    # --- Helper: CDF SSE Calculation ---
    def calc_ecdf_sse(cdf_func, data, params=None):
        try:
            sorted_data = np.sort(data)
            n_samples = len(data)
            y_emp = np.arange(1, n_samples + 1) / n_samples
            
            if params:
                y_fit = cdf_func(sorted_data, *params)
            else:
                y_fit = cdf_func(sorted_data)
                
            return (1/len(y_emp)) *  np.sum((y_emp - y_fit) ** 2)
        except Exception:
            return np.nan

    dist_names = [
        'gamma', 'normal', 'exponential', 'lognormal', 'chi_square',
        'rayleigh', 'weibull', 'logistic', 'gumbel', 'laplace', 'trimodal'
    ]
    
    param_ranges = {
        'gamma': lambda: {'a': np.random.uniform(0.5, 5), 'scale': np.random.uniform(0.5, 3)},
        'normal': lambda: {'loc': np.random.uniform(-5, 5), 'scale': np.random.uniform(0.5, 3)},
        'expo': lambda: {'scale': np.random.uniform(0.5, 3)},
        'lognorm': lambda: {'s': np.random.uniform(0.1, 1.5), 'scale': np.random.uniform(0.5, 3)},
        'chi2': lambda: {'df': np.random.randint(1, 11)},
        'rayleigh': lambda: {'scale': np.random.uniform(0.5, 3)},
        'weibull': lambda: {'c': np.random.uniform(0.5, 3), 'scale': np.random.uniform(0.5, 3)},
        'logistic': lambda: {'loc': np.random.uniform(-5, 5), 'scale': np.random.uniform(0.5, 3)},
        'gumbel': lambda: {'loc': np.random.uniform(-5, 5), 'scale': np.random.uniform(0.5, 3)},
        'laplace': lambda: {'loc': np.random.uniform(-5, 5), 'scale': np.random.uniform(0.5, 3)},
        'trimodal': lambda: {
            'loc1': np.random.uniform(-5, -1), 'loc2': np.random.uniform(-1, 1), 
            'loc3': np.random.uniform(1, 5), 'scale': np.random.uniform(0.3, 1.0)
        }
    }

    print(f"Starting Benchmark: {n_trials} trials per distribution, {m} samples each.")
    print(f"Comparing Bezier(n={n}) vs Beta, Pearson, Johnson, GenGamma.")
    print("=" * 80)

    all_results = []
    start_total = time.time()
    
    total_iterations = n_trials * len(dist_names)
    pbar = tqdm(total=total_iterations, desc="Benchmarking", unit="fit")
    counter = 0

    for trial in range(1, n_trials + 1):
        
        current_params = {k: v() for k, v in param_ranges.items()}

        for name in dist_names:
            
            if name == 'trimodal':
                p = current_params['trimodal']
                data = np.concatenate([
                    np.random.normal(loc=p['loc1'], scale=p['scale'], size=m//3),
                    np.random.normal(loc=p['loc2'], scale=p['scale'], size=m//3),
                    np.random.normal(loc=p['loc3'], scale=p['scale'], size=m - 2*(m//3))
                ])
            else:
                mapping = {
                    'gamma': (stats.gamma, {'a': current_params['gamma']['a'], 'scale': current_params['gamma']['scale']}),
                    'normal': (stats.norm, {'loc': current_params['normal']['loc'], 'scale': current_params['normal']['scale']}),
                    'exponential': (stats.expon, {'scale': current_params['expo']['scale']}),
                    'lognormal': (stats.lognorm, {'s': current_params['lognorm']['s'], 'scale': current_params['lognorm']['scale']}),
                    'chi_square': (stats.chi2, {'df': current_params['chi2']['df']}),
                    'rayleigh': (stats.rayleigh, {'scale': current_params['rayleigh']['scale']}),
                    'weibull': (stats.weibull_min, {'c': current_params['weibull']['c'], 'scale': current_params['weibull']['scale']}),
                    'logistic': (stats.logistic, {'loc': current_params['logistic']['loc'], 'scale': current_params['logistic']['scale']}),
                    'gumbel': (stats.gumbel_r, {'loc': current_params['gumbel']['loc'], 'scale': current_params['gumbel']['scale']}),
                    'laplace': (stats.laplace, {'loc': current_params['laplace']['loc'], 'scale': current_params['laplace']['scale']}),
                }
                func, args = mapping[name]
                data = func.rvs(**args, size=m)

            # --- A. Bezier Fits ---
            methods = [('Bez_ProjGrad', 'projgrad'), ('Bez_Nonlinear', 'nonlinear'), ('Bez_NelderMead', 'neldermead')]
            for method_label, method_key in methods:
                t_start = time.time()
                sse = np.nan
                try:
                    distfit = DistFit(data=data, n=n)
                    if method_key == 'projgrad':
                        bezierv, mse = distfit.fit(method='projgrad', step_size_PG=0.001, max_iter_PG=1000, threshold_PG=1e-4)
                    elif method_key == 'nonlinear':
                        bezierv, mse = distfit.fit(method='nonlinear', solver_NL='ipopt')
                    else:
                        bezierv, mse = distfit.fit(method='neldermead', max_iter_NM=200)
                    
                    if hasattr(bezierv, 'cdf'):
                        sse = calc_ecdf_sse(bezierv.cdf, data)
                except: pass
                all_results.append({'Trial': trial, 'Dist': name, 'Method': method_label, 'Time': time.time() - t_start, 'SSE': mse})

            # --- B. Benchmarks ---
            
            # Generalized Beta
            t_start = time.time()
            sse = np.nan
            try:
                p_beta = stats.beta.fit(data)
                sse = calc_ecdf_sse(stats.beta.cdf, data, p_beta)
            except: pass
            all_results.append({'Trial': trial, 'Dist': name, 'Method': 'Gen_Beta', 'Time': time.time() - t_start, 'SSE': sse})

            # Pearson III
            t_start = time.time()
            sse = np.nan
            try:
                p_p3 = stats.pearson3.fit(data)
                sse = calc_ecdf_sse(stats.pearson3.cdf, data, p_p3)
            except: pass
            all_results.append({'Trial': trial, 'Dist': name, 'Method': 'Pearson3', 'Time': time.time() - t_start, 'SSE': sse})

            # Johnson
            t_start = time.time()
            sse = np.nan
            try:
                p_sb = stats.johnsonsb.fit(data)
                sse_sb = calc_ecdf_sse(stats.johnsonsb.cdf, data, p_sb)
                p_su = stats.johnsonsu.fit(data)
                sse_su = calc_ecdf_sse(stats.johnsonsu.cdf, data, p_su)
                sse = min(sse_sb if not np.isnan(sse_sb) else np.inf, sse_su if not np.isnan(sse_su) else np.inf)
                if sse == np.inf: sse = np.nan
            except: pass
            all_results.append({'Trial': trial, 'Dist': name, 'Method': 'Johnson', 'Time': time.time() - t_start, 'SSE': sse})

            # 4-Param
            t_start = time.time()
            sse = np.nan
            try:
                p_gg = stats.gengamma.fit(data)
                sse = calc_ecdf_sse(stats.gengamma.cdf, data, p_gg)
            except: pass
            all_results.append({'Trial': trial, 'Dist': name, 'Method': '4Param_GG', 'Time': time.time() - t_start, 'SSE': sse})

        
            counter += 1
            pbar.update(1)
           

    
    pbar.close()

    print("\nProcessing Results...")
    df = pd.DataFrame(all_results)
    df.to_csv(output_file, index=False)
    
    summary = df.groupby(['Dist', 'Method']).agg({
        'SSE': ['mean'],
        'Time': ['mean']
    }).reset_index()
    summary.columns = ['Dist', 'Method', 'Mean_SSE', 'Mean_Time']
    
    print("\nSummary of Average Performance (Lower SSE is better):")
    print("-" * 100)
    print(f"{'Distribution':<15} | {'Method':<15} | {'Mean SSE':<12} | {'Time (s)':<8}")
    print("-" * 100)
    
    summary = summary.sort_values(by=['Dist', 'Mean_SSE'])
    for _, row in summary.iterrows():
        print(f"{row['Dist']:<15} | {row['Method']:<15} | {row['Mean_SSE']:12.6e} | {row['Mean_Time']:8.4f}")

    return summary

In [ ]:
summary = test_bezierv_fit_distributions(m=1000, n=10, n_trials=50, output_file='benchmark_results.csv')

Starting Benchmark: 50 trials per distribution, 1000 samples each.
Comparing Bezier(n=10) vs Beta, Pearson, Johnson, GenGamma.


Benchmarking:   1%|          | 4/550 [03:11<6:28:16, 42.67s/fit]